## Streamlit month UI

Preferred interactive viewer (farm parquet only — no EnergyPlus):

```powershell
cd vibe_code_apps_22
$env:LAKESIDE_SITE_ROOT="C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"
streamlit run eplus_gym_app\streamlit_app.py --server.port 8765
```

Open http://localhost:8765 — pick month + strategies. Grow farm with
`python -u scripts/run_eplus_gym_month_farm.py --months 2026-01,2026-02 --dry-run`
(then `--execute` overnight when ready).


# Lakeside E+ gym — results viewer

**Safe pattern:** run the CLI (outside Jupyter), then open this notebook to plot artifacts.

`powershell
python -u scripts\run_eplus_gym_rules.py --mode lookup
# live E+ (NOT in notebook — ctypes callbacks crash Cursor/Jupyter):
# python -u scripts\run_eplus_gym_rules.py --mode live --epw PATH.epw --idf PATH.idf
`

Artifacts: 
eports/eplus_gym/  
Honesty: STRUCTURAL_LOAD_DIAGNOSTIC · lookup = FARM_LOOKUP_EMULATOR · promote=False

Do **not** call 
un_rule_episode(..., mode="live") from this notebook.


In [ ]:
%matplotlib inline
import json, os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = Path('..').resolve()
if not (ROOT / 'eplus_gym').is_dir():
    ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

OUT = ROOT / 'reports' / 'eplus_gym'
FIG = OUT / 'figures'
CARD = OUT / 'rule_dr_scorecard.json'
print('OUT', OUT)
print('card exists', CARD.is_file())


## 1 · Scorecard from last CLI run


In [ ]:
if not CARD.is_file():
    display(Markdown(
        '**No scorecard yet.** Run:\n\n'
        'python -u scripts/run_eplus_gym_rules.py --mode lookup\n\n'
        'then re-run this cell.'
    ))
else:
    card = json.loads(CARD.read_text(encoding='utf-8'))
    summary = pd.DataFrame(card.get('strategies') or [])
    display(summary)
    display(Markdown(f"_note: {card.get('note','')}_"))
    overlay = FIG / 'rule_dr_overlay.png'
    if overlay.is_file():
        display(Image(filename=str(overlay)))
        print('figure', overlay)
    else:
        print('no overlay png yet')


## 2 · Re-plot trajectories from parquet (no EnergyPlus)


In [ ]:
trajs = sorted(OUT.glob('traj_*.parquet'))
if not trajs:
    display(Markdown('_No 	raj_*.parquet under reports/eplus_gym — run the CLI first._'))
else:
    colors = {
        'baseline': '#264653', 'flat_24_7': '#6c757d', 'deep_setback': '#2a9d8f',
        'stagger_preheat': '#e9c46a', 'morning_all_on': '#e76f51',
    }
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    days = set()
    for pq in trajs:
        df = pd.read_parquet(pq)
        # traj_{strategy}_{day}.parquet
        parts = pq.stem.split('_')
        # stem like traj_baseline_2026-01-09
        sid = pq.stem.replace('traj_', '', 1)
        day = None
        if len(sid) > 10 and sid[-10:].count('-') == 2:
            day = sid[-10:]
            sid = sid[:-11]
        days.add(day)
        c = colors.get(sid, '#333333')
        t_h = df['step'].to_numpy(dtype=float) / 4.0 if 'step' in df.columns else range(len(df))
        if 'facility_kw' in df.columns:
            axes[0].plot(t_h, df['facility_kw'], color=c, lw=1.9, label=sid)
        if 'htg_sp_f' in df.columns:
            axes[1].plot(t_h, df['htg_sp_f'], color=c, lw=1.5, label=sid)
    axes[0].set_ylabel('facility kW')
    axes[0].set_title(f"E+ gym artifacts · days={sorted(d for d in days if d)}")
    axes[0].legend(loc='best', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    axes[1].set_ylabel('htg SP °F')
    axes[1].set_xlabel('hour')
    axes[1].legend(loc='best', fontsize=8)
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    out_png = FIG / 'notebook_viewer_overlay.png'
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=120)
    plt.show()
    print('saved', out_png)
    print('loaded', [p.name for p in trajs])


## 3 · Next UI options (not in this notebook)

| Option | Pros | Cons |
|---|---|---|
| **CLI + this viewer** (now) | Stable; no ctypes in Jupyter | Manual refresh |
| **FastAPI + Plotly** | Browser charts; trigger lookup jobs via HTTP | Extra process; still keep live E+ off the request thread |
| **egui desktop (archived)** | Native app | Hybrid ONNX path archived — would rebuild for gym artifacts |
| **Live E+ in notebook** | Feels interactive | Crashes Cursor (your traceback) — **avoid** |

Recommended next neat step: tiny FastAPI that shells 
un_eplus_gym_rules.py --mode lookup and serves Plotly from parquet.
